# Pipeline Flow

This notebook, `Admission_Eligibility_Checker_using_KB.ipynb`, reads combined Markdown summaries produced by `doc_executiveSummary.ipynb` (files matching `document_summaries*.md` in each applicant folder), then writes `admission_eligibility_checker_result.json`.

`doc_reconciliationAgent.ipynb` separately reconciles per-document JSON summaries from multiple model-output folders. This eligibility checker does **not** currently read the reconciliation notebook's JSON or CSV outputs. Its input is the executive-summary Markdown, so reconciliation is not part of this notebook's current data path.

# Admission Eligibility Checker (Knowledge-Base Driven)

FAIR-VID / DocAnalysis pipeline notebook

## Where Google Drive directories are
This notebook runs in Colab and uses this path:
- /content/drive/MyDrive/colab-output/dream_applicant_application

Example applicant folder used for this documentation:
- /colab-output/dream_applicant_application/18190_27580

## Purpose
For each applicant folder, the notebook:
1. Loads rules and prompt templates from the FAIR-VID GitHub repository.
2. Reads applicant markdown document summaries from the applicant folder.
3. Detects study level from short_application_info.json (Bachelor or Master).
4. Builds a prompt and sends it to Gemini.
5. Saves one structured result (**admission_eligibility_checker_result.json**) JSON in the same applicant folder.

## Exact input/output contract (based on folder 18190_27580)

### Files this notebook reads as input from each applicant folder
Required (at least one):
- Any markdown files matching *.md

In the 18190_27580 example, detected markdown inputs were:
- combined_stage2_results.md
- document summaries_gemma4_e4bcombined_stage2_results.md
- document_summaries___glm-ocr_granite418bcombined_stage2_results.md

Optional metadata file:
- short_application_info.json

From short_application_info.json, this notebook uses:
- awards_abbr

In the 18190_27580 example, short_application_info.json contains keys:
- awards_abbr
- awards_full
- citizenship
- name

Note: this notebook currently reads only awards_abbr from this file.


### File this notebook writes as output per applicant
- admission_eligibility_checker_result.json


If model output is not valid JSON or an exception occurs, the notebook still writes admission_eligibility_checker_result.json with fallback/error payload fields.

## Processing flow
1. Mount Google Drive.
2. Download Bachelor/Master rules and international grading knowledge bases.
3. Download Bachelor/Master prompt templates.
4. Iterate applicant directories under dream_applicant_application.
5. Read all *.md files and optional short_application_info.json.
6. Build prompt with the correct rules branch.
7. Run Gemini generation.
8. Save admission_eligibility_checker_result.json in the applicant folder.

## Configuration
Environment requirement: one of these API keys must be set.
- GOOGLE_API_KEY (preferred)
- GENAI_API_KEY
- GEMINI_API_KEY

## Operational notes
- The notebook processes all applicant directories under the root path, not only one folder.


## Mount Google Drive


In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

Mounting Google Drive...
Mounted at /content/drive


## Load Knowledge Bases
Downloads Bachelor and Master admission rules, plus international grading/requirement dictionaries, from the FAIR-VID prompt repository.

In [ ]:
import requests

url_bachelor_admission_rules = "https://raw.githubusercontent.com/fair-vid/admission_prompts/main/admissions-knowledge-base/requirements_and_grading/bachelor_admission_rules.json"
url_international_bachelor_requirements_and_grading = "https://raw.githubusercontent.com/fair-vid/admission_prompts/main/admissions-knowledge-base/requirements_and_grading/international_bachelor_requirements_and_grading.json"

url_master_admission_rules = "https://raw.githubusercontent.com/fair-vid/admission_prompts/main/admissions-knowledge-base/requirements_and_grading/master_admission_rules.json"
international_master_requirements_and_grading = "https://raw.githubusercontent.com/fair-vid/admission_prompts/main/admissions-knowledge-base/requirements_and_grading/international_master_requirements_and_grading.json"

data_bachelor = requests.get(url_bachelor_admission_rules).json()
data_intl_bachelor = requests.get(url_international_bachelor_requirements_and_grading).json()

data_master = requests.get(url_master_admission_rules).json()
data_intl_master = requests.get(international_master_requirements_and_grading).json()

print(f"All 4 dictionaries. bachelor: {len(data_bachelor)} {len(data_intl_bachelor)}       master: {len(data_master)} {len(data_intl_master)}")


All 4 dictionaries. bachelor: 8 51       master: 8 42


## Load Prompt Templates
Fetches the Bachelor and Master eligibility prompt templates used later for dynamic prompt construction.

In [ ]:
url_Bachelor_Admission_Eligibility_Checker_Prompt = "https://raw.githubusercontent.com/fair-vid/admission_prompts/main/document_analysis/Bachelor_Admission_Eligibility_Checker_Prompt.txt"
Bachelor_Admission_Eligibility_Checker_Prompt = requests.get(url_Bachelor_Admission_Eligibility_Checker_Prompt).text

url_Master_Admission_Eligibility_Checker_Prompt = "https://raw.githubusercontent.com/fair-vid/admission_prompts/main/document_analysis/Master_Admission_Eligibility_Checker_Prompt.txt"
Master_Admission_Eligibility_Checker_Prompt = requests.get(url_Master_Admission_Eligibility_Checker_Prompt).text

## Inspect Prompt Content
Prints the Master prompt template so you can verify instructions and placeholders before processing applicants.

In [ ]:
print(Bachelor_Admission_Eligibility_Checker_Prompt)

You are an admissions requirements checking assistant for Bachelor programme applications.

Your task is to assess whether the applicant appears to meet the Bachelor admission requirements for the selected programme.

You must use ONLY the information provided in:
1. General Bachelor admission rules KB
2. Country-specific secondary education requirements and grading KB
3. Application/programme information
4. Structured applicant/student information
5. Applicant document info
6. Deterministic pre-check results, if provided

Do NOT use external knowledge.
Do NOT invent missing facts.
Do NOT assume that a missing document exists.
Do NOT assume English proficiency unless explicitly shown.
Do NOT make the final official admissions decision; provide an evidence-based recommendation only.

Important Bachelor admission logic:
- If a requirement is clearly satisfied, mark it PASS.
- If a requirement is clearly not satisfied, mark it FAIL.
- If information is missing or unclear, mark it INSUFFIC

## Define Core Utilities and Model Client
Defines helper utilities for JSON extraction, award-level detection, markdown loading, prompt filling, and Gemini response generation.

In [ ]:
from pathlib import Path
import json
import os
import re
import time
import pandas as pd

try:
    from google import genai
    from google.genai import types
except Exception:
    genai = None
    types = None

try:
    from google.colab import userdata
except Exception:
    userdata = None

MODEL = "gemini-flash-lite-latest"
DRIVE_DIR = Path("/content/drive/MyDrive/colab-output/dream_applicant_application")

# Set this to a TSV path to process only selected applications.
FILTER_FILE = DRIVE_DIR / "FILTERED_LIST_Applicant_Application_ids_OneApp.tsv" # Leave as an empty string "" to process all application folders.


if userdata is not None:
    api_key = userdata.get("GOOGLE_API_KEY")
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key


def extract_json_from_response(response_text):
    if not response_text:
        return False, {"raw_response": response_text, "parse_status": "empty_response"}, ".json"

    match = re.search(r"```json\s*\n(.*?)\n```", response_text, re.DOTALL | re.IGNORECASE)
    candidate_text = match.group(1).strip() if match else response_text.strip()

    try:
        parsed = json.loads(candidate_text)
        return True, parsed, ".json"
    except json.JSONDecodeError:
        return False, {"raw_response": response_text, "parse_status": "non_json_response"}, ".json"


def normalize_awards_abbr(value):
    normalized = str(value or "Bachelor").strip().lower()
    if "master" in normalized:
        return "Master"
    return "Bachelor"


def read_awards_abbr(app_dir):
    info_path = app_dir / "short_application_info.json"
    if not info_path.exists():
        return "Bachelor"

    try:
        with open(info_path, "r", encoding="utf-8") as f:
            info = json.load(f)
        return normalize_awards_abbr(info.get("awards_abbr", "Bachelor"))
    except Exception as exc:
        print(f"  Could not read {info_path.name}: {exc}")
        return "Bachelor"


def read_applicant_document_info(app_dir):
    md_files = sorted([f for f in app_dir.glob("*.md") if f.name.startswith("document_summaries")])
    if not md_files:
        return "", []

    parts = []
    for md_file in md_files:
        try:
            md_text = md_file.read_text(encoding="utf-8")
        except Exception as exc:
            md_text = f"[Could not read file: {exc}]"
        parts.append(f"# {md_file.name}\n{md_text}")

    return "\n\n---\n\n".join(parts), md_files


def extract_country_codes_from_md(md_text):
    """Extract unique country codes from markdown text using regex pattern \"country_code\": \"XX\"."""
    pattern = r'"country_code"\s*:\s*"([A-Z]{2,3})"'
    codes = re.findall(pattern, md_text)
    return sorted(set(codes))


def build_filtered_country_rule_kb(intl_data, country_codes):
    """
    Filter international requirements dict to only the extracted country codes.
    Codes not found in intl_data are explicitly flagged as NOT FOUND.
    Returns a compact JSON string to keep prompt size small.
    """
    if not country_codes:
        return json.dumps(
            {"note": "No country_code values found in applicant documents. No country-specific rules applied."},
            ensure_ascii=False,
            indent=2,
        )

    found = {}
    not_found = []

    for code in country_codes:
        if code in intl_data:
            found[code] = intl_data[code]
        else:
            not_found.append(code)

    result = {}
    if found:
        result["country_requirements"] = found
    if not_found:
        result["not_found_country_codes"] = {
            code: "NOT FOUND: No specific requirements data available for this country code."
            for code in not_found
        }

    return json.dumps(result, ensure_ascii=False, indent=2)


def discover_subdirs(base_dir, filter_file):
    base_path = Path(base_dir)
    if not base_path.exists():
        print(f"ERROR: Base directory does not exist: {base_dir}")
        return []

    if filter_file and Path(filter_file).exists():
        print(f"Found filter file: {filter_file}")
        try:
            filter_df = pd.read_csv(filter_file, sep="\t")
            allowed_subdirs = set(filter_df["applicant_application"].astype(str).tolist())
            subdirs = [d.name for d in base_path.iterdir() if d.is_dir() and d.name in allowed_subdirs]
            print(f"Filtered to {len(subdirs)} subdirectories based on TSV.")
            return subdirs
        except Exception as exc:
            print(f"ERROR reading {filter_file}: {exc}")
            return []

    if filter_file:
        print(f"Filter file not found: {filter_file}. Processing all subdirectories.")
    else:
        print("No filter file provided. Processing all subdirectories.")

    return [d.name for d in base_path.iterdir() if d.is_dir()]


def fill_prompt(
    template,
    *,
    application_info,
    student_info,
    precheck_results,
    rules_kb,
    country_rule_kb,
    applicant_document_info,
    rules_placeholder,
):
    prompt = template
    replacements = {
        "{{APPLICATION_INFO}}": application_info,
        "{{STUDENT_INFO}}": student_info,
        "{{PRECHECK_RESULTS}}": precheck_results,
        "{{COUNTRY_RULE}}": country_rule_kb,
        "{{APPLICANT_DOCUMENT_INFO}}": applicant_document_info,
        rules_placeholder: rules_kb,
    }

    for key, value in replacements.items():
        prompt = prompt.replace(key, value)

    return prompt


def generate_with_gemini(filled_prompt, model=MODEL):
    if genai is None or types is None:
        raise RuntimeError("google.genai is not available. Install google-genai and set GOOGLE_API_KEY.")

    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GENAI_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Missing GOOGLE_API_KEY, GENAI_API_KEY, or GEMINI_API_KEY.")

    client = genai.Client(api_key=api_key)
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=filled_prompt)])]
    result_text = ""

    for chunk in client.models.generate_content_stream(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(),
    ):
        if getattr(chunk, "text", None):
            result_text += chunk.text

    return result_text

## Run Batch Eligibility Processing
Scans applicant folders, selects the Bachelor or Master path, executes model inference, and writes one result JSON per applicant (or structured error payload on failure).

In [ ]:
if not DRIVE_DIR.exists():
    raise FileNotFoundError(f"Drive directory not found: {DRIVE_DIR}")

print(f"Scanning applicant folders under {DRIVE_DIR}")
subdirs = discover_subdirs(DRIVE_DIR, FILTER_FILE)
if not subdirs:
    print("No subdirectories to process. Exiting.")
else:
    processed_applicants = 0

    for app_dir_name in subdirs:
        app_dir = DRIVE_DIR / app_dir_name
        applicant_document_info, md_files = read_applicant_document_info(app_dir)
        if not md_files:
            print(f"  Skipping {app_dir.name}: no md files found")
            continue

        # Extract country codes from MD content and build a filtered KB
        country_codes = extract_country_codes_from_md(applicant_document_info)

        awards_abbr = read_awards_abbr(app_dir)
        if awards_abbr == "Master":
            prompt_template = Master_Admission_Eligibility_Checker_Prompt
            rules_kb = json.dumps(data_master, ensure_ascii=False, indent=2)
            country_rule_kb = build_filtered_country_rule_kb(data_intl_master, country_codes)
            rules_placeholder = "{{MASTER_RULES}}"
        else:
            prompt_template = Bachelor_Admission_Eligibility_Checker_Prompt
            rules_kb = json.dumps(data_bachelor, ensure_ascii=False, indent=2)
            country_rule_kb = build_filtered_country_rule_kb(data_intl_bachelor, country_codes)
            rules_placeholder = "{{BACHELOR_RULES}}"

        print(f"  {app_dir.name} | {awards_abbr} | country codes: {country_codes or '(none found)'}")

        filled_prompt = fill_prompt(
            prompt_template,
            application_info="No additional information available.",
            student_info="No additional information available.",
            precheck_results="No additional information available.",
            rules_kb=rules_kb,
            country_rule_kb=country_rule_kb,
            applicant_document_info=applicant_document_info,
            rules_placeholder=rules_placeholder,
        )

        print(f"    Prompt size: {len(filled_prompt)} chars  ({len(md_files)} md file(s))")
        with open("DEBUG_PROMPT.txt", "w", encoding="utf-8") as debug_file:
            debug_file.write(filled_prompt)

        output_path = app_dir / "admission_eligibility_checker_result.json"
        try:
            response_text = generate_with_gemini(filled_prompt, model=MODEL)
            is_json, content, _ = extract_json_from_response(response_text)

            if is_json and isinstance(content, dict):
                content.setdefault("awards_abbr", awards_abbr)
                content.setdefault("source_markdown_files", [f.name for f in md_files])
                content.setdefault("application_root", app_dir.name)
                content.setdefault("detected_country_codes", country_codes)
            else:
                content = {
                    "awards_abbr": awards_abbr,
                    "application_root": app_dir.name,
                    "source_markdown_files": [f.name for f in md_files],
                    "detected_country_codes": country_codes,
                    "response": content,
                }

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(content, f, ensure_ascii=False, indent=2)

            print(f"    Saved {output_path.name}")
            processed_applicants += 1
            time.sleep(1)

        except Exception as exc:
            error_payload = {
                "awards_abbr": awards_abbr,
                "application_root": app_dir.name,
                "source_markdown_files": [f.name for f in md_files],
                "detected_country_codes": country_codes,
                "error": str(exc),
            }
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(error_payload, f, ensure_ascii=False, indent=2)
            print(f"    Saved error payload to {output_path.name}: {exc}")

    print(f"Done. Processed {processed_applicants} applicant folder(s).")

Scanning applicant folders under /content/drive/MyDrive/colab-output/dream_applicant_application
Found filter file: /content/drive/MyDrive/colab-output/dream_applicant_application/FILTERED_LIST_Applicant_Application_ids_OneApp.tsv
Filtered to 1 subdirectories based on TSV.
  29696_25487 | Master | country codes: ['LR']
    Prompt size: 17337 chars  (1 md file(s))
    Saved admission_eligibility_checker_result.json
Done. Processed 1 applicant folder(s).


In [ ]:
print(prompt_template )


You are an admissions requirements checking assistant for Master programme applications.

Your task is to assess whether the applicant appears to meet the admission requirements for the selected Master programme.

You must use ONLY the information provided in:
1. General Master admission rules KB
2. Country-specific requirements and grading KB
3. Application/programme information
4. Structured applicant/student information
5. Applicant document info
6. Deterministic pre-check results

Do NOT use external knowledge.
Do NOT invent missing facts.
Do NOT assume that a missing document exists.
Do NOT assume English proficiency unless explicitly shown.
Do NOT make the final official admissions decision; provide an evidence-based recommendation only.

Important admission logic:
- If a requirement is clearly satisfied, mark it PASS.
- If a requirement is clearly not satisfied, mark it FAIL.
- If information is missing or unclear, mark it INSUFFICIENT_INFORMATION.
- If the requirement depends o